# <font style="font-family:roboto;color:#455e6c"> Generating datasets for Machine Learning Interatomic Potentials with the ASSYST method </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 16 March 2025 </font> </br> </br>
Marvin Poul, Sarath Menon, Haitham Gaafer, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

### References

- [Poul, M., Huber, L. & Neugebauer, J. Automated Generation of Structure Datasets for Machine Learning Potentials and Alloys. Preprint](https://doi.org/10.21203/rs.3.rs-4732459/v1) (2024)

In [1]:
from pyiron_core import Workflow, PyironFlow, as_function_node, as_inp_dataclass_node
# from pyiron_nodes.atomistic.mlips.fitting.assyst import make_assyst
# from pyiron_core.pyiron_nodes.atomistic.assyst2.structures import ElementInput, Stoichiometry, StoichiometryTable

In [2]:
from collections.abc import Sequence
from dataclasses import dataclass, field
from itertools import product
from functools import lru_cache

import pandas as pd
from ase import Atoms

In [3]:
@dataclass(frozen=True)
class Stoichiometry(Sequence):
    stoichiometry: tuple[dict[str, int]]

    @property
    def elements(self) -> set[str]:
        """Set of elements present in stoichiometry."""
        e = set()
        for s in self.stoichiometry:
            s = e.union(s.keys())
        return s

    # FIXME: Self only availabe in >=3.11
    def __add__(self, other: "Stoichiometry") -> "Stoichiometry":
        """Extend underlying list of stoichiometries."""
        return Stoichiometry(self.stoichiometry + other.stoichiometry)

    def __or__(self, other: "Stoichiometry") -> "Stoichiometry":
        """Inner product of underlying stoichiometries.

        Must not share elements with other stoichiometry."""
        assert self.elements.isdisjoint(
            other.elements
        ), "Can only or stoichiometries of different elements!"
        s = ()
        for me, you in zip(self.stoichiometry, other.stoichiometry, strict=False):
            s += (me | you,)
        return Stoichiometry(s)

    def __mul__(self, other: "Stoichiometry") -> "Stoichiometry":
        """Outer product of underlying stoichiometries.

        Must not share elements with other stoichiometry."""
        assert self.elements.isdisjoint(
            other.elements
        ), "Can only multiply stoichiometries of different elements!"
        s = ()
        for me, you in product(self.stoichiometry, other.stoichiometry):
            s += (me | you,)
        return Stoichiometry(s)

    # Sequence Impl'
    def __getitem__(self, index: int) -> dict[str, int]:
        return self.stoichiometry[index]

    def __len__(self) -> int:
        return len(self.stoichiometry)


@as_function_node
def ElementInput(
    element: str,
    min_ion: int = 1,
    max_ion: int = 10,
    step_ion: int = 1,
) -> Stoichiometry:
    stoichiometry = Stoichiometry(
        tuple({element: i} for i in range(min_ion, max_ion + 1, step_ion))
    )
    return stoichiometry


@as_function_node("df")
def StoichiometryTable(stoichiometry: Stoichiometry) -> pd.DataFrame:
    return pd.DataFrame(stoichiometry.stoichiometry)



## <font style="font-family:roboto;color:#455e6c"> Background </font> 

*Automated Small SYmetric Structure Training* or ASSYST is a method to generate training data for machine learning potentials.
The key idea is to use small structures to automatically explore structurally and chemically diverse atomic environments and provide training data around the energetically most favorable ones.

### <font style="font-family:roboto;color:#455e6c"> Workflow Overview </font>

![image](img/AssystSchematic.svg)

### <font style="font-family:roboto;color:#455e6c"> Transferability </font>

ASSYST trained potentials describe also structures that they are not directly trained on, such as point and planar defects.

![image](img/Fig8_MTP24_2d0_8d2_DefectsManual.png)

Liquid state is also well described and potentials are stable for long running thermodynamic integrations.

![image](img/Fig11_MgCa.png)

This phase diagram is our goal for today!

### <font style="font-family:roboto;color:#455e6c"> Literature </font>

- Mg and Defects: https://journals.aps.org/prb/abstract/10.1103/PhysRevB.107.104103
- Ternary Mg/Al/Ca: https://www.researchsquare.com/article/rs-4732459/v1

## <font style="font-family:roboto;color:#455e6c"> Constructing element combinations </font> 

The first step in the ASSYST workflow is to decide which chemical space to cover and how densely.
Increasing the new number of total atoms allows you to generate more and more complex structures
and also sample the chemical space more densely.

Here's an example for a ternary system, where we sampled the unaries with ASSYST datasets of 1-10 Atoms and the binaries and ternaries with 2-8 or 3-8 Atoms, respectively.

![img](img/Fig3_Everything_Conc_Plot.png)

Log-histogram of composition of a final training set.

`Elements` wraps a list of compositions at which we will sample random crystals.

In [4]:
mg = Stoichiometry((
    {'Mg': 2}, {'Mg': 4}
))
mg

Stoichiometry(stoichiometry=({'Mg': 2}, {'Mg': 4}))

In [5]:
al = Stoichiometry((
    {'Al': 1}, {'Al': 2}
))
al

Stoichiometry(stoichiometry=({'Al': 1}, {'Al': 2}))

Can be combined with standard python operations.

In [6]:
mg + al

Stoichiometry(stoichiometry=({'Mg': 2}, {'Mg': 4}, {'Al': 1}, {'Al': 2}))

In [7]:
mg | al

Stoichiometry(stoichiometry=({'Mg': 2, 'Al': 1}, {'Mg': 4, 'Al': 2}))

In [8]:
mg * al

Stoichiometry(stoichiometry=({'Mg': 2, 'Al': 1}, {'Mg': 2, 'Al': 2}, {'Mg': 4, 'Al': 1}, {'Mg': 4, 'Al': 2}))

Created by the ElementInput node and visualized by StoichiometryTable.

In [9]:
wf = Workflow("ASSYST_Elements_Unary")
wf.Element = ElementInput(element="Mg")
wf.ElementsTable = StoichiometryTable(wf.Element)

In [10]:
pf = PyironFlow([wf])
pf.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>

Build a small workflow that creates a table with Mg, Al and Ca so that:
1. Mg:Ca is always 2:1
2. combines it with 2-8 Al
3. has at least 2 Ca in every composition
4. contains at most 16 Atoms

Check `utilities` for nodes to `Add()`, `Multiply()` or `Or()` objects together.

Check `atomistic` -> `mlips` -> `fitting` -> `assyst` for nodes to `FilterSize()` or `ElementsTable()`
</div>

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Solution</b></p>
    Load this workflow for the solution
</div>

In [11]:
@as_function_node("or")
def Or(x, y):
    return x | y

In [12]:
@as_function_node("product")
def Multiply(x, y):
    return x * y

In [13]:
@as_function_node("filtered")
def FilterSize(
    elements: Stoichiometry,
    min: int | None = 0,
    max: int | None = None,
):
    """Filter an Elements object by size.

    Args:
        min (int): new object has at least this number of atoms
        max (int): new object has at most this number of atoms

    Returns:
        Elements: filtered object
    """
    import math
    if max is None:
        max = math.inf
    return Stoichiometry(tuple(s for s in elements
                                if min <= sum(s.values()) <= max ))

In [14]:
wf = Workflow("ASSYST_Elements_Combine")
wf.Mg = ElementInput(element="Mg", min_ion=4, max_ion=10, step_ion=2)
wf.Ca = ElementInput(element="Ca", min_ion=2, max_ion=10, step_ion=2)
wf.Al = ElementInput(element="Al", min_ion=1, max_ion=10, step_ion=1)
wf.combo = Or(x=wf.Mg, y=wf.Ca)
wf.product = Multiply(x=wf.combo, y=wf.Al)
wf.filter = FilterSize(elements=wf.product, min=0, max=16)
wf.ElementsTable = StoichiometryTable(wf.filter)

In [15]:
pf = PyironFlow([wf])
pf.gui

## <font style="font-family:roboto;color:#455e6c"> Full Workflow for a Small Structure Set </font> 

This demonstration uses the GRACE universal force fields for the relaxation steps.
Usually we would run them in low convergence DFT.

In [16]:
from assyst.crystals import pyxtal
from dataclasses import dataclass
from ase.atoms import Atoms
from ase.constraints import FixAtoms
from ase.filters import StrainFilter, FrechetCellFilter
from pyiron_core import Node
import numpy as np
# from pyiron_core.pyiron_nodes.atomistic.assyst2.plot import PlotSPG

In [17]:
@as_function_node("plot", use_cache=False)
def PlotSPG(structures: list[Atoms]):
    """Plot a histogram of space groups in input list."""
    import matplotlib.pyplot as plt
    from structuretoolkit.analyse import get_symmetry

    spacegroups = []
    for structure in structures:
        spacegroups.append(get_symmetry(structure).info["number"])
    plt.hist(spacegroups)
    return plt.show()

In [18]:
from enum import Enum

class RelaxMode(Enum):
    VOLUME = "volume"
    # CELL = "cell"
    INTERNAL = "internal"
    FULL = "full"

    def apply_filter_and_constraints(self, structure):
        match self:
            case RelaxMode.VOLUME:
                structure.set_constraint(FixAtoms(np.ones(len(structure),dtype=bool)))
                return FrechetCellFilter(structure, hydrostatic_strain=True)
            case RelaxMode.INTERNAL:
                return structure
            case RelaxMode.FULL:
                return FrechetCellFilter(structure)
            case _:
                raise ValueError("Lazy Marvin")

In [19]:
@as_function_node
def Relax(mode: str | RelaxMode, calculator: Node, opt: Node, structure: Atoms) -> Atoms:
    from ase.optimize import LBFGS
    from ase.calculators.singlepoint import SinglePointCalculator

    if isinstance(mode, str):
        mode = mode.lower()
    mode = RelaxMode(mode)

    structure = structure.copy()

    # FIXME: meh
    match mode:
        case RelaxMode.VOLUME:
            calculator.inputs.use_symmetry = True
            structure.calc = calculator.pull()
        case RelaxMode.FULL | RelaxLoop.INTERNAL:
            calculator.inputs.use_symmetry = False
            structure.calc = calculator.pull()
        case _:
            assert False

    filtered_structure = mode.apply_filter_and_constraints(structure)
    lbfgs = LBFGS(filtered_structure, logfile="/dev/null")
    lbfgs.run(fmax=opt.inputs.force_tolerance.value, steps=opt.inputs.max_steps.value)
    calc = structure.calc
    structure.calc = SinglePointCalculator(structure, **{
            'energy': calc.get_potential_energy(),
            'forces': calc.get_forces(),
            'stress': calc.get_stress()
    })
    relaxed_structure = structure
    relaxed_structure.constraints.clear()
    return relaxed_structure

In [20]:
@as_function_node
def SpaceGroupSampling(
        elements: Stoichiometry,
        spacegroups: list[int] | tuple[int,...] | None = None,
        max_atoms: int = 10,
        max_structures: int | None = None,
) -> list[Atoms]:
    """
    Create symmetric random structures.

    Args:
        elements (Elements): list of compositions per structure
        spacegroups (list of int): which space groups to generate
        max_atoms (int): do not generate structures larger than this
        max_structures (int): generate at most this many structures
    Returns:
        list of Atoms: generated structures
    """
    from warnings import catch_warnings
    from tqdm.auto import tqdm
    import math

    if spacegroups is None:
        spacegroups = list(range(1,231))
    if max_structures is None:
        max_structures = math.inf

    structures = []
    with catch_warnings(category=UserWarning, action='ignore'):
        for stoich in (bar := tqdm(elements)):
            elements, num_atomss = zip(*stoich.items())
            stoich_str = "".join(f"{s}{n}" for s, n in zip(elements, num_atomss))
            bar.set_description(stoich_str)
            structures += [s['atoms'] for s in pyxtal(spacegroups, elements, num_atomss)]
            if len(structures) > max_structures:
                structures = structures[:max_structures]
                break
    return structures


In [21]:
def rattle(structure: Atoms, sigma: float) -> Atoms:
    """Randomly displace positions with gaussian noise.

    Operates INPLACE."""
    structure.rattle(stdev=sigma)
    return structure


def stretch(structure: Atoms, hydro: float, shear: float) -> Atoms:
    """Randomly stretch cell with uniform noise.

    Operates INPLACE."""
    strain = shear * (2 * np.random.rand(3, 3) - 1)
    strain = 0.5 * (strain + strain.T)  # symmetrize
    np.fill_diagonal(strain, 1 + hydro * (2 * np.random.rand(3) - 1))
    structure.set_cell(structure.cell.array @ strain, scale_atoms=True)
    return structure


@as_function_node
def Rattle(structure: Atoms, sigma: float, samples: int) -> list[Atoms]:
    structures = []
    # no point in rattling single atoms
    if len(structure) > 1:
        for _ in range(samples):
            structures.append(
                    stretch(
                        rattle(structure.copy(), sigma),
                        hydro=0.05, shear=0.005
                    )
            )
    return structures


@as_function_node
def RattleLoop(
        structures: list[Atoms], sigma: float, samples: int
) -> list[Atoms]:
    rattled_structures = []
    for structure in structures:
        rattled_structures += Rattle(structure, sigma, samples).pull()
    return rattled_structures


@as_function_node
def Stretch(
        structure: Atoms, hydro: float, shear: float, samples: int,
        hydro_shear_ratio: float = 0.7
) -> list[Atoms]:
    structures = []
    for _ in range(samples):
        if np.random.rand() < hydro_shear_ratio:
            ihydro, ishear = hydro, 0.05
        else:
            ihydro, ishear = 0.05, shear
        structures.append(
                stretch(structure.copy(), hydro=ihydro, shear=ishear)
        )
    return structures


@as_function_node
def StretchLoop(
        structures: list[Atoms], hydro: float, shear: float, samples: int,
        hydro_shear_ratio: float = 0.7
) -> list[Atoms]:
    stretched_structures = []
    for structure in structures:
        stretched_structures += Stretch(
                structure, hydro, shear, samples, hydro_shear_ratio
        ).pull()
    return stretched_structures

In [22]:
def make_assyst(
        name, *elements,
        min_atoms=2, max_atoms=4, max_structures=50,
        delete_existing_savefiles=False
):
    wf = Workflow(name)

    element_nodes = []
    if len(elements) > 0:
        e1, *elements = elements
        stoi = ElementInput(e1, min_atoms=min_atoms, max_atoms=max_atoms)
        setattr(wf, 'Element_1', stoi)
        element_nodes.append(stoi)
        for i, e in enumerate(elements):
            en = ElementInput(e, min_atoms=min_atoms, max_atoms=max_atoms)
            setattr(wf, f'Element_{i+2}', en)
            element_nodes.append(en)
            stoi = Multiply(stoi, en)
            setattr(wf, f'Multiply_{i+1}_{i+2}', stoi)
        if len(elements) > 0:
            wf.Multiply = stoi
        spg = SpaceGroupSampling(
                elements=stoi,
                spacegroups=None,
                max_atoms=len(element_nodes) * max_atoms,
                max_structures=max_structures
        )
    else:
        spg = SpaceGroupSampling(
                max_atoms=max_atoms,
                max_structures=max_structures
        )
    plotspg = PlotSPG(spg)

    calc_config = Grace()
    optimizer_settings = GenericOptimizerSettings()

    volume_relax = RelaxLoop(mode="volume", calculator=calc_config,
                             opt=optimizer_settings, structures=spg)
    full_relax = RelaxLoop(mode="full", calculator=calc_config,
                           opt=optimizer_settings,
                           structures=volume_relax.outputs.relaxed_structures)

    rattle = RattleLoop(
            structures=full_relax.outputs.relaxed_structures,
            sigma=0.25,
            samples=4
    )

    stretch = StretchLoop(
            structures=full_relax.outputs.relaxed_structures,
            hydro=0.8,
            shear=0.2,
            samples=4
    )

    combine_structures = CombineStructures(
            spg,
            volume_relax.outputs.relaxed_structures,
            full_relax.outputs.relaxed_structures,
            rattle,
            stretch
    )

    savestructures = SaveStructures(combine_structures, "data/Structures_Everything")

    wf.SpaceGroupSampling = spg
    wf.PlotSPG = plotspg

    wf.Calculator = calc_config
    wf.OptimizerSettings = optimizer_settings
    wf.VolumeRelax = volume_relax
    wf.FullRelax = full_relax

    wf.Rattle = rattle
    wf.Stretch = stretch

    wf.CombineStructures = combine_structures
    wf.SaveStructures = savestructures
    return wf

In [23]:
@as_function_node
@lru_cache
def Grace(model: str = "GRACE-FS-OAM",  use_symmetry=True):
    """Universal Graph Atomic Cluster Expansion models."""
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
    from tensorpotential.calculator import grace_fm
    grace_obj = grace_fm(model)
    return grace_obj

In [24]:
@as_inp_dataclass_node
@dataclass
class GenericOptimizerSettings:
    max_steps: int = 10
    force_tolerance: float = 1e-2

In [25]:
@as_function_node
def CombineStructures(
        spacegroups: list[Atoms] | None,
        volume_relax: list[Atoms] | None,
        full_relax: list[Atoms] | None,
        rattle: list[Atoms] | None,
        stretch: list[Atoms] | None,
) -> list[Atoms]:
    """Combine individual structure sets into a full training set."""
    from functools import reduce
    structures = [spacegroups, volume_relax, full_relax, rattle, stretch]
    structures = reduce(list.__add__, (s or [] for s in structures), [])
    if len(structures) == 0:
        logging.warn("Either no inputs given or all inputs are empty. "
                     "Returning the empty list!")
    return structures

In [26]:
@as_function_node
def SaveStructures(
        structures: list[Atoms],
        filename: str
):
    """Save list of structures into a pickled dataframe.

    Columns are:
        'name': a structure label
        'ase_atoms': the ASE object for the actual structure
        'number_of_atoms': the number of atoms inside the structure

    If `filename` does not end with 'pckl.gz', it is added.

    Args:
        structures (list of Atoms): structures to save
        filename (str): path where the dataframe is written to
    """
    import pandas as pd
    import os.path
    df = pd.DataFrame([
        {'name': s.info.get('label', f'structure_{i}'),
         'ase_atoms': s,
         'number_of_atoms': len(s),
         } for i, s in enumerate(structures)])
    if not filename.endswith("pckl.gz"):
        filename += ".pckl.gz"
    dirname = os.path.dirname(filename)
    os.makedirs(dirname, exist_ok=True)
    df.to_pickle(filename)
    return df

In [27]:
@as_function_node
def RelaxLoop(
        mode: str | RelaxMode,
        calculator: Node,
        opt: Node,
        structures: list[Atoms]
) -> list[Atoms]:
    from tqdm.auto import tqdm
    mode = RelaxMode(mode)
    relaxed_structures = []
    for structure in tqdm(structures, desc=f"Relax {mode.value}"):
        relaxed_structures.append(
                Relax(mode, calculator, opt, structure).pull()
        )
    return relaxed_structures

In [28]:
wf = make_assyst('ASSYST', 'Mg', 'Ca', 'Al')

wrapped:  <function Grace at 0x7f0bdd719bc0>


In [29]:
pf = PyironFlow([wf])

implement connect to self
implement connect to self
implement connect to self
implement connect to self
connected to node (self)
connected to node (self)
connected to node (self)
connected to node (self)


In [30]:
pf.gui

## <font style="font-family:roboto;color:#455e6c"> Precomputed Full Workflow with Large Structure Set </font> 

This is the same workflow, but pre-run with realistic input for a Unary system.
It contains ~10k structures and you can attach plotting functions at various nodes to view them.

In [31]:
wf = make_assyst('ASSYST_Mg_FULL', 'Mg', min_atoms=1, max_atoms=10, max_structures=None)

wrapped:  <function Grace at 0x7f0bdd719bc0>


In [32]:
pf = PyironFlow([wf])

implement connect to self
implement connect to self
implement connect to self
implement connect to self
connected to node (self)
connected to node (self)
connected to node (self)
connected to node (self)


In [33]:
pf.gui

<img src="img/logo_roll.png" width="1200">